# Modul 6 · Schätzen und Testen
**Data Science 1** · linkit courses

Wie genau kennst du die Störungsrate einer Maschine? Arbeitet M-03 auf Sollmaß, und lässt sich zeigen, dass M-09 es tut? Wie viele Lieferungen bräuchte das Werk, um einen Unterschied zwischen L-05 und L-01 nachzuweisen? In Modul 5 hast du solche Fragen mit Simulationen beantwortet. Hier rechnest du Konfidenzintervalle und Tests mit Formeln und mit `scipy.stats`, planst die Teststärke und gehst mit vielen Tests auf einmal um.

**Lernziele**
1. Schätzer nach Verzerrung, Streuung und Konsistenz beurteilen
2. Konfidenzintervalle für Mittelwerte und Anteile mit Formel berechnen und deuten
3. die Logik eines Tests mit Fehlern erster und zweiter Art erklären
4. den passenden Test wählen: t, Welch, Mann-Whitney, Anteile, Chi-Quadrat
5. Teststärke und nötige Stichprobengröße vor einer Untersuchung bestimmen
6. bei vielen Tests Bonferroni, Holm oder Benjamini-Hochberg anwenden
7. statistische Signifikanz von praktischer Relevanz unterscheiden und Ergebnisse vollständig berichten

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import von SciPy dauert im Browser einige Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 4, TODO 6 bis 8 sowie TODO 10 bis 13. TODO 5, TODO 9 und die Bonusaufgaben vertiefen.
- Jede TODO-Zelle mit Zufall legt ihren eigenen Generator mit festem Seed an, `rng = np.random.default_rng(...)`. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg besteht.
- Gib Ergebnisse von SciPy mit `float(...)` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["lieferungen.csv", "messwerte.csv", "stoerungen.csv", "produktion.csv", "maschinen.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Schätzer und ihre Güte
*Lerneinheit 1 · Skript Abschnitt 1*

Ein **Schätzer** $\hat\theta$ ist eine Rechenvorschrift, die aus einer Stichprobe eine Zahl für einen unbekannten Parameter $\theta$ macht. Er ist selbst eine Zufallsvariable. Seine Güte beschreiben

$$\mathrm{Bias} = E(\hat\theta) - \theta, \qquad \mathrm{MSE} = E\big[(\hat\theta - \theta)^2\big] = \mathrm{Bias}^2 + \mathrm{Var}(\hat\theta).$$

Erwartungstreu heißt Bias 0, konsistent heißt: Mit wachsendem $n$ liegt der Schätzer mit gegen 1 gehender Wahrscheinlichkeit beliebig nah an $\theta$. Neu ist SciPy, das Paket importierst du als `from scipy import stats`. Die Hilfsfunktion `zeige` druckt Zahlen mit 6 gültigen Stellen, so stehen im Browser und in Colab dieselben Ziffern da.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
lieferungen = pd.read_csv("daten/lieferungen.csv")
messwerte = pd.read_csv("daten/messwerte.csv", dtype={"schicht": SCHICHT_TYP})
stoerungen = pd.read_csv("daten/stoerungen.csv", dtype={"schicht": SCHICHT_TYP})
produktion = pd.read_csv("daten/produktion.csv", dtype={"schicht": SCHICHT_TYP})
maschinen = pd.read_csv("daten/maschinen.csv")
verzug_l01 = lieferungen.loc[lieferungen["lieferant"] == "L-01", "verzug_h"]
verzug_l05 = lieferungen.loc[lieferungen["lieferant"] == "L-05", "verzug_h"]
m03 = messwerte.loc[messwerte["maschine"] == "M-03", "durchmesser_mm"]
m09 = messwerte.loc[messwerte["maschine"] == "M-09", "durchmesser_mm"]
print(len(verzug_l01), len(verzug_l05), len(m03), len(m09))


def zeige(*werte):
    """Druckt Zahlen mit 6 gültigen Stellen, damit im Browser und in Colab dieselben Ziffern stehen. None bleibt None."""
    def text(w):
        if w is None or isinstance(w, (str, bool)):
            return str(w)
        if np.ndim(w) == 0:
            return f"{float(w):.6g}"
        return "[" + " ".join(f"{float(v):.6g}" for v in np.ravel(w)) + "]"
    print(*[text(w) for w in werte])

### TODO 1 · Verzerrung der Varianz per Simulation
Die 2.826 Durchmesser von M-09 sind hier die Grundgesamtheit. Ihre Varianz mit Teiler $n$ steht schon in `sigma2_pop`.
1. `proben_m09`: 100.000 Stichproben mit je 4 Werten aus `m09`, mit Zurücklegen gezogen, als Array der Form (100000, 4). Tipp: Indizes mit `rng.integers` ziehen und damit `m09.to_numpy()` auswählen.
2. `verh_n` und `verh_n1`: der Mittelwert der Stichprobenvarianzen mit Teiler $n$ (`ddof=0`) und mit Teiler $n - 1$ (`ddof=1`), jeweils geteilt durch `sigma2_pop`.
3. `mse_n` und `mse_n1`: der mittlere quadratische Fehler beider Schätzer, $E\big[(s^2 - \sigma^2)^2\big]$, geteilt durch `sigma2_pop ** 2`.

In [ ]:
rng = np.random.default_rng(1)
sigma2_pop = m09.var(ddof=0)

# TODO 1a
proben_m09 = None

# TODO 1b und 1c
verh_n = None
verh_n1 = None
mse_n = None
mse_n1 = None

zeige(verh_n, verh_n1, mse_n, mse_n1)

In [ ]:
check("TODO 1a Stichproben", lambda: proben_m09.shape == (100_000, 4) and np.isin(proben_m09[:1000], m09.to_numpy()).all())
check("TODO 1b Verzerrung", lambda: abs(verh_n - 0.75) <= 0.012 and abs(verh_n1 - 1.0) <= 0.012)
check("TODO 1c mittlerer quadratischer Fehler", lambda: 0.43 <= mse_n <= 0.47 and 0.64 <= mse_n1 <= 0.73 and mse_n < mse_n1)

Mit Teiler $n$ liegt die Schätzung im Mittel bei nur 75 % der wahren Varianz, genau $(n - 1)/n$. Mit $n - 1$ trifft sie. Der mittlere quadratische Fehler ist aber beim verzerrten Schätzer kleiner, etwa 0,45 gegen 0,69: Ein wenig Bias kann sich lohnen, wenn er viel Streuung spart.

**Maximum Likelihood.** Für die Poisson-Rate ist die Log-Likelihood $\ell(\lambda) = \sum_i (x_i \ln\lambda - \lambda - \ln x_i!)$. Ihr Maximum liegt beim Mittelwert $\hat\lambda = \bar x$, der Standardfehler ist $\sqrt{\hat\lambda/n}$. Für M-03 wie im Skript, mit allen Schichten:

In [ ]:
alle_schichten = pd.MultiIndex.from_frame(produktion[["datum", "schicht"]].drop_duplicates())
st_m03 = stoerungen[stoerungen["maschine"] == "M-03"]
zaehl_m03 = st_m03.groupby(["datum", "schicht"], observed=True).size().reindex(alle_schichten, fill_value=0)
raster = np.linspace(0.7, 1.1, 401)
loglik = np.array([stats.poisson.logpmf(zaehl_m03, lam).sum() for lam in raster])
lam_m03 = zaehl_m03.mean()
se_lam_m03 = math.sqrt(lam_m03 / len(zaehl_m03))
print(round(raster[loglik.argmax()], 3), round(lam_m03, 4), round(se_lam_m03, 4))
print(round(lam_m03 - 1.96 * se_lam_m03, 3), round(lam_m03 + 1.96 * se_lam_m03, 3))

### TODO 2 · Poisson-Rate von M-09 mit Standardfehler
1. `zaehl_m09`: die Zahl der Störungen an M-09 je Schicht, mit `alle_schichten` als Index und 0 für Schichten ohne Störung. Dazu `lam_m09` (der Schätzer $\hat\lambda$), `se_lam_m09` (sein Standardfehler) und `ki_lam_m09`, das 95-%-Intervall $\hat\lambda \pm 1{,}96\,\mathrm{SE}$ als Array mit zwei Werten.
2. `loglik_m09`: die Log-Likelihood für jeden Wert in `raster_m09`, und `lam_max_m09`: der Wert aus `raster_m09`, bei dem sie am größten ist.

In [ ]:
raster_m09 = np.linspace(0.3, 0.7, 401)

# TODO 2a
zaehl_m09 = None
lam_m09 = None
se_lam_m09 = None
ki_lam_m09 = None

# TODO 2b
loglik_m09 = None
lam_max_m09 = None

zeige(lam_m09, se_lam_m09, ki_lam_m09, lam_max_m09)

In [ ]:
check("TODO 2a Rate und Standardfehler", lambda: len(zaehl_m09) == 1447 and int(zaehl_m09.sum()) == 736
      and math.isclose(lam_m09, 0.508639, abs_tol=1e-6) and math.isclose(se_lam_m09, 0.0187489, abs_tol=1e-6))
check("TODO 2a Intervall", lambda: len(ki_lam_m09) == 2 and math.isclose(ki_lam_m09[0], 0.471891, abs_tol=1e-5)
      and math.isclose(ki_lam_m09[1], 0.545387, abs_tol=1e-5))
check("TODO 2b Maximum der Log-Likelihood", lambda: len(loglik_m09) == 401 and math.isclose(lam_max_m09, 0.509, abs_tol=1e-9))

M-09 hat 0,509 Störungen je Schicht, das Intervall reicht von 0,472 bis 0,545. Das Intervall von M-03 liegt mit 0,835 bis 0,932 weit darüber, die Raten unterscheiden sich deutlich. Beide Intervalle setzen das Poisson-Modell voraus.

## 2 · Konfidenzintervalle mit Formel
*Lerneinheit 2 · Skript Abschnitt 2*

Für einen Mittelwert aus $n$ Werten ist das 95-%-Intervall $\bar x \pm t_{n-1;\,0{,}975}\,s/\sqrt n$ (Student 1908). Das Quantil liefert `stats.t.ppf(0.975, n - 1)`, für großes $n$ geht es in $z_{0{,}975} = 1{,}96$ über. `stats.ttest_1samp(x, mu0).confidence_interval()` rechnet dasselbe Intervall. Für einen Anteil $\hat p = k/n$ ist das **Wilson-Intervall** besser als das Wald-Intervall $\hat p \pm z\sqrt{\hat p(1-\hat p)/n}$:

$$\frac{\hat p + \frac{z^2}{2n}}{1 + \frac{z^2}{n}} \pm \frac{z}{1 + \frac{z^2}{n}}\,\sqrt{\frac{\hat p\,(1 - \hat p)}{n} + \frac{z^2}{4n^2}}.$$

In SciPy rechnet es `stats.binomtest(k, n).proportion_ci(method="wilson")`. Die Quantile und die Märzwoche von M-03 und M-09 wie im Skript:

In [ ]:
for fg in [4, 9, 29, 99, 999]:
    print(fg, round(float(stats.t.ppf(0.975, fg)), 3))
print("Normal", round(float(stats.norm.ppf(0.975)), 3))

woche = messwerte[messwerte["datum"].between("2026-03-02", "2026-03-06")]
for maschine in ["M-03", "M-09"]:
    messung = woche.loc[woche["maschine"] == maschine, "durchmesser_mm"]
    t_krit = stats.t.ppf(0.975, len(messung) - 1)
    halbe = t_krit * messung.std() / math.sqrt(len(messung))
    ki = stats.ttest_1samp(messung, 25.0).confidence_interval()
    print(maschine, len(messung), round(messung.mean(), 4), round(messung.std(), 4),
          round(messung.mean() - halbe, 4), round(messung.mean() + halbe, 4), round(float(ki.low), 4), round(float(ki.high), 4))

### TODO 3 · t-Intervall von Hand
Eine andere Woche: M-03 vom 13. bis 17. Januar 2025.
1. `kw3`: die Durchmesser von M-03 in dieser Woche. `mittel_kw3`, `se_kw3` (Standardfehler) und `t_krit_kw3` (das Quantil der t-Verteilung). `ki_hand`: das 95-%-Intervall als Array `[unten, oben]`.
2. `ki_scipy`: dasselbe Intervall mit `stats.ttest_1samp(...).confidence_interval()`, als Array `[unten, oben]`. `p_kw3`: der zweiseitige p-Wert des t-Tests gegen 25,000 mm.
3. `ki_z`: das Intervall mit 1,96 statt des t-Quantils.

In [ ]:
# TODO 3a
kw3 = None
mittel_kw3 = None
se_kw3 = None
t_krit_kw3 = None
ki_hand = None

# TODO 3b und 3c
ki_scipy = None
p_kw3 = None
ki_z = None

zeige(ki_hand, ki_scipy, ki_z, p_kw3)

In [ ]:
check("TODO 3a Woche und Standardfehler", lambda: len(kw3) == 30 and math.isclose(mittel_kw3, 25.0037, abs_tol=1e-7)
      and math.isclose(se_kw3, 0.00183963, abs_tol=1e-7) and math.isclose(t_krit_kw3, 2.04523, abs_tol=1e-5))
check("TODO 3a Intervall von Hand", lambda: math.isclose(ki_hand[0], 24.9999375, abs_tol=1e-7) and math.isclose(ki_hand[1], 25.0074625, abs_tol=1e-7))
check("TODO 3b wie SciPy", lambda: np.allclose(ki_scipy, ki_hand, rtol=0, atol=1e-9) and math.isclose(p_kw3, 0.053673, abs_tol=1e-5))
check("TODO 3c mit z statt t", lambda: math.isclose(ki_z[0], 25.0000943, abs_tol=1e-7) and ki_z[0] > 25.0 > ki_hand[0])

Das t-Intervall reicht von 24,99994 bis 25,00746 mm und enthält das Sollmaß knapp, $p = 0{,}054$. Mit 1,96 läge die untere Grenze bei 25,00009 mm, und du hättest eine Abweichung gemeldet. Bei 30 Werten ist der Unterschied zwischen $z$ und $t$ klein, an der Grenze entscheidet er.

### TODO 4 · Wald gegen Wilson
1. `quoten`: ein DataFrame mit einer Zeile je Lieferant (Index `L-01` bis `L-06`) und den Spalten `k` (Mängel), `n` (geprüfte Teile), `p` (Quote), `wald_unten`, `wald_oben`, `wilson_unten` und `wilson_oben`. Wilson rechnest du mit `stats.binomtest(int(k), int(n)).proportion_ci(method="wilson")`.
2. `wald_0` und `wilson_0`: die beiden 95-%-Intervalle für eine Lieferung mit 0 Mängeln unter 80 geprüften Teilen, je als Array `[unten, oben]`.
3. `deckung_wald` und `deckung_wilson`: Wie oft treffen beide Intervalle die wahre Quote $p = 0{,}0104$ bei $n = 80$? Simuliere 100.000 Stichproben mit `rng.binomial` und rechne beide Intervalle vektorisiert, wie im Skript.

In [ ]:
z975 = stats.norm.ppf(0.975)

# TODO 4a
quoten = None

# TODO 4b
wald_0 = None
wilson_0 = None

# TODO 4c
rng = np.random.default_rng(4)
deckung_wald = None
deckung_wilson = None

print(None if quoten is None else quoten.round(5).to_string())
zeige(wald_0, wilson_0, deckung_wald, deckung_wilson)

In [ ]:
check("TODO 4a Form", lambda: list(quoten.index) == ["L-01", "L-02", "L-03", "L-04", "L-05", "L-06"]
      and {"k", "n", "p", "wald_unten", "wald_oben", "wilson_unten", "wilson_oben"} <= set(quoten.columns)
      and int(quoten.loc["L-05", "k"]) == 13 and int(quoten.loc["L-05", "n"]) == 1830)
check("TODO 4a Wald", lambda: math.isclose(quoten.loc["L-05", "wald_unten"], 0.0032559, abs_tol=1e-6)
      and math.isclose(quoten.loc["L-01", "wald_oben"], 0.0111305, abs_tol=1e-6))
check("TODO 4a Wilson", lambda: math.isclose(quoten.loc["L-05", "wilson_unten"], 0.0041562, abs_tol=1e-6)
      and math.isclose(quoten.loc["L-05", "wilson_oben"], 0.0121164, abs_tol=1e-6))
check("TODO 4b keine Mängel", lambda: np.allclose(wald_0, [0, 0]) and wilson_0[0] < 1e-12 and math.isclose(wilson_0[1], 0.0458181, abs_tol=1e-6))
check("TODO 4c Überdeckung", lambda: abs(deckung_wald - 0.5665) <= 0.007 and abs(deckung_wilson - 0.9488) <= 0.004)

Für die großen Lieferanten liegen Wald und Wilson fast gleich, bei L-05 mit 13 Mängeln unter 1.830 Teilen verschiebt Wilson das Intervall nach oben, auf 0,42 % bis 1,21 %. Für eine Lieferung ohne Mangel liefert Wald $[0, 0]$, Wilson 0 bis 4,6 %. Bei $p = 1{,}04\,\%$ und 80 Teilen trifft Wald nur in etwa 57 % der Fälle, Wilson in fast 95 %.

## 3 · Die Logik des Tests
*Lerneinheit 3 · Skript Abschnitt 3*

1. Nullhypothese $H_0$ und Alternative $H_1$ festlegen.
2. Teststatistik wählen und ihre Verteilung unter $H_0$ bestimmen.
3. p-Wert: Wahrscheinlichkeit, unter $H_0$ eine mindestens so extreme Statistik zu sehen.
4. Ist $p \le \alpha$, wird $H_0$ verworfen. Der **Fehler erster Art** (verwerfen, obwohl $H_0$ stimmt) passiert mit Wahrscheinlichkeit $\alpha$, der **Fehler zweiter Art** (beibehalten, obwohl $H_1$ stimmt) mit $\beta$.

Das 95-%-Intervall enthält $\mu_0$ genau dann, wenn der zweiseitige t-Test zum Niveau 5 % nicht verwirft. Das Beispiel von Hand und der A/A-Vergleich aus dem Skript: Die Lieferungen von L-01 werden 2.000-mal zufällig halbiert, und etwa 5 % der Welch-Tests werden „signifikant“.

In [ ]:
hand = np.array([6.0, 2.0, 9.0, 4.0, 4.0])
t_hand = hand.mean() / (hand.std(ddof=1) / math.sqrt(len(hand)))
p_hand = 2 * stats.t.sf(abs(t_hand), df=len(hand) - 1)
test_hand = stats.ttest_1samp(hand, 0.0)
print(round(t_hand, 3), round(float(p_hand), 4), round(float(test_hand.statistic), 3), round(float(test_hand.pvalue), 4))
print(round(float(stats.ttest_1samp(hand, 0.0, alternative="greater").pvalue), 4))

pop = verzug_l01.to_numpy()
rng = np.random.default_rng(4)
R_aa = 2000
reihen = rng.permuted(np.tile(np.arange(len(pop)), (R_aa, 1)), axis=1)
aa = stats.ttest_ind(pop[reihen[:, :590]], pop[reihen[:, 590:]], axis=1, equal_var=False)
print(aa.pvalue.shape, round((aa.pvalue < 0.05).mean(), 4), round(math.sqrt(0.05 * 0.95 / R_aa), 4))

### TODO 5 · Intervall und Test für alle Wochen
In `messwerte_kw` steht zu jeder Messung die Kalenderwoche nach ISO 8601.
1. `wochen`: ein DataFrame mit einer Zeile je Maschine und Woche und den Spalten `maschine`, `kw`, `n`, `p` (p-Wert des t-Tests gegen 25,000 mm), `unten` und `oben` (95-%-Intervall).
2. `dual_ok`: `True`, wenn in jeder Zeile gilt: $p < 0{,}05$ genau dann, wenn das Intervall 25,000 mm nicht enthält.
3. `signif_woche`: eine Series mit den Maschinen als Index und der Zahl der Wochen mit $p < 0{,}05$.

In [ ]:
messwerte_kw = messwerte.assign(kw=pd.to_datetime(messwerte["datum"]).dt.strftime("%G-W%V"))

# TODO 5
wochen = None
dual_ok = None
signif_woche = None

print(None if wochen is None else wochen.head().round(6).to_string())
print(dual_ok, None if signif_woche is None else signif_woche.to_dict())

In [ ]:
check("TODO 5 Wochen", lambda: len(wochen) == 202 and {"maschine", "kw", "n", "p", "unten", "oben"} <= set(wochen.columns)
      and int(wochen["n"].sum()) == 5652)
check("TODO 5 Dualität", lambda: dual_ok is True and int(((wochen["unten"] > 25) | (wochen["oben"] < 25)).sum()) == 38)
check("TODO 5 Wochen mit p < 0,05", lambda: int(signif_woche["M-03"]) == 33 and int(signif_woche["M-09"]) == 5)

In jeder der 202 Wochen passen Intervall und Test zusammen. M-03 weicht in 33 von 101 Wochen nachweisbar ab, obwohl sie über alle Messungen klar über dem Sollmaß liegt: Mit 30 Messungen je Woche ist die Teststärke gering. Bei M-09 sind es 5 von 101 Wochen, etwa so viele, wie der Fehler erster Art erwarten lässt, wenn sie im Mittel genau auf Sollmaß arbeitet.

## 4 · Die wichtigsten Tests
*Lerneinheit 4 · Skript Abschnitt 4*

| Frage | Test | SciPy |
|---|---|---|
| Mittelwert gegen Sollwert | Einstichproben-t-Test | `stats.ttest_1samp(x, mu0)` |
| zwei Mittelwerte | Welch-Test | `stats.ttest_ind(a, b, equal_var=False)` |
| Werte einer Gruppe tendenziell größer | Mann-Whitney-U | `stats.mannwhitneyu(a, b)` |
| zwei Anteile | z-Test, Chi-Quadrat, Fisher | `stats.chi2_contingency(tafel, correction=False)`, `stats.fisher_exact(tafel)` |
| zwei Merkmale zusammenhängend | Chi-Quadrat-Unabhängigkeitstest | `stats.chi2_contingency(kreuztabelle)` |
| vorher und nachher | gepaarter t-Test | `stats.ttest_rel(vor, nach)` |

Welch rechnet mit $t = (\bar x_1 - \bar x_2)/\sqrt{s_1^2/n_1 + s_2^2/n_2}$ und den Freiheitsgraden

$$\nu = \frac{\big(s_1^2/n_1 + s_2^2/n_2\big)^2}{\frac{(s_1^2/n_1)^2}{n_1 - 1} + \frac{(s_2^2/n_2)^2}{n_2 - 1}}.$$

L-03 gegen L-01 wie im Skript:

In [ ]:
verzug_l03 = lieferungen.loc[lieferungen["lieferant"] == "L-03", "verzug_h"]
welch_l03 = stats.ttest_ind(verzug_l03, verzug_l01, equal_var=False)
student_l03 = stats.ttest_ind(verzug_l03, verzug_l01, equal_var=True)
print(round(verzug_l03.mean() - verzug_l01.mean(), 2), round(verzug_l03.std(), 2), round(verzug_l01.std(), 2))
print(round(float(welch_l03.statistic), 2), round(float(welch_l03.df), 1), f"{float(welch_l03.pvalue):.1e}")
print(round(float(student_l03.statistic), 2), round(float(student_l03.df)), f"{float(student_l03.pvalue):.1e}")

### TODO 6 · Welch-Test von Hand
Vergleiche den Verpackungslieferanten L-06 mit L-01.
1. `verzug_l06`: die Verzüge von L-06. `t_welch` und `df_welch`: Teststatistik und Freiheitsgrade nach den Formeln oben, `p_welch`: der zweiseitige p-Wert mit `stats.t.sf`.
2. `welch_scipy`: das Ergebnis von `stats.ttest_ind(..., equal_var=False)` zum Vergleich.

In [ ]:
# TODO 6a
verzug_l06 = None
t_welch = None
df_welch = None
p_welch = None

# TODO 6b
welch_scipy = None

zeige(t_welch, df_welch, p_welch)

In [ ]:
check("TODO 6a Welch von Hand", lambda: len(verzug_l06) == 628 and math.isclose(t_welch, 0.761585, abs_tol=1e-5)
      and math.isclose(df_welch, 1091.02, abs_tol=0.01) and math.isclose(p_welch, 0.446472, abs_tol=1e-5))
check("TODO 6b wie SciPy", lambda: math.isclose(t_welch, float(welch_scipy.statistic), rel_tol=1e-9)
      and math.isclose(df_welch, float(welch_scipy.df), rel_tol=1e-9) and math.isclose(p_welch, float(welch_scipy.pvalue), rel_tol=1e-7))

L-06 kommt im Mittel 0,55 Stunden später als L-01, $p = 0{,}45$: kein Nachweis. Von Hand und mit SciPy kommt dasselbe heraus, die Freiheitsgrade liegen mit 1.091 deutlich unter $n_1 + n_2 - 2 = 1.805$, weil L-06 stärker streut und weniger Lieferungen hat.

### TODO 7 · L-05 gegen L-01 mit vier Tests
1. `perm_p(a, b, R, rng)`: der zweiseitige Permutationstest für die Differenz der Mittelwerte aus Modul 5, aber vektorisiert. Mische in Blöcken von höchstens 1.000 Permutationen mit `rng.permuted(np.tile(zusammen, (block, 1)), axis=1)` und gib den p-Wert $(b + 1)/(R + 1)$ zurück.
2. `vergleich_l05`: ein DataFrame mit dem Index `Student`, `Welch`, `Mann-Whitney`, `Permutation` und der Spalte `p`, alle zweiseitig. Für den Permutationstest nimmst du `R = 10_000`.

In [ ]:
# TODO 7a
def perm_p(a, b, R, rng):
    return None            # ersetze diese Zeile

# TODO 7b
rng = np.random.default_rng(7)
vergleich_l05 = None

print(None if vergleich_l05 is None else vergleich_l05.round(4))

In [ ]:
probe_p = None
try:
    probe_p = perm_p([6.0, 10.0], [1.0, 3.0], 3000, np.random.default_rng(0))
except Exception:
    pass
check("TODO 7a perm_p am Mini-Beispiel", lambda: abs(probe_p - 1 / 3) <= 4 * math.sqrt(2 / 9 / 3000) + 1 / 3001)
check("TODO 7b t-Tests", lambda: list(vergleich_l05.index) == ["Student", "Welch", "Mann-Whitney", "Permutation"]
      and math.isclose(vergleich_l05.loc["Student", "p"], 0.202348, abs_tol=1e-5) and math.isclose(vergleich_l05.loc["Welch", "p"], 0.289484, abs_tol=1e-5))
check("TODO 7b Mann-Whitney", lambda: math.isclose(vergleich_l05.loc["Mann-Whitney", "p"], 0.625744, abs_tol=1e-5))
check("TODO 7b Permutation", lambda: 0.17 <= vergleich_l05.loc["Permutation", "p"] <= 0.22)

In [ ]:
if vergleich_l05 is None:
    print("⏳ Erst TODO 7 lösen.")
else:
    print(f"Streuung L-05: {verzug_l05.std():.2f} h, L-01: {verzug_l01.std():.2f} h")
    print(f"Mediane: L-05 {verzug_l05.median():.1f} h, L-01 {verzug_l01.median():.1f} h; Mittelwerte: {verzug_l05.mean():.2f} h und {verzug_l01.mean():.2f} h")

Student und Permutationstest liegen bei etwa 0,2, Welch bei 0,29, Mann-Whitney bei 0,63. Student und der Permutationstest unterstellen unter $H_0$ gleiche Verteilungen, also auch gleiche Streuung. L-05 streut stärker, Students gemeinsame Varianz ist für L-05 zu klein und der p-Wert zu klein. Welch prüft den mittleren Verzug ohne diese Annahme. Mann-Whitney fragt, ob Lieferungen von L-05 tendenziell später kommen, und das tun sie kaum: Der Unterschied steckt in wenigen langen Verzügen. Keiner der vier Tests weist einen Unterschied nach.

**Anteile.** Der z-Test für zwei Anteile und der Chi-Quadrat-Test der 2×2-Tafel ohne Stetigkeitskorrektur sind gleichwertig, $\chi^2 = z^2$. L-03 gegen L-01 wie im Skript:

In [ ]:
mangel = lieferungen.groupby("lieferant")[["mangel_stueck", "geprueft_stueck"]].sum()
k_l03, n_l03 = mangel.loc["L-03"]
k_l01, n_l01 = mangel.loc["L-01"]
p_gemeinsam = (k_l03 + k_l01) / (n_l03 + n_l01)
z_quote = (k_l03 / n_l03 - k_l01 / n_l01) / math.sqrt(p_gemeinsam * (1 - p_gemeinsam) * (1 / n_l03 + 1 / n_l01))
tafel = np.array([[k_l03, n_l03 - k_l03], [k_l01, n_l01 - k_l01]])
chi2_tafel = stats.chi2_contingency(tafel, correction=False)
print(round(k_l03 / n_l03, 4), round(k_l01 / n_l01, 4), round(z_quote, 2), round(z_quote ** 2, 2), round(float(chi2_tafel.statistic), 2))
print(f"{2 * stats.norm.sf(abs(z_quote)):.1e}", f"{float(chi2_tafel.pvalue):.1e}", f"{float(stats.fisher_exact(tafel).pvalue):.1e}")

### TODO 8 · Mangelquote L-02 gegen L-04
1. `z_l02_l04` und `p_z_l02_l04`: der z-Test für die Mangelquoten von L-02 und L-04 von Hand, mit gemeinsamer Quote unter $H_0$ und zweiseitigem p-Wert (`stats.norm.sf`).
2. `chi2_l02_l04`: das Ergebnis von `stats.chi2_contingency` für die 2×2-Tafel aus mangelhaften und mangelfreien Teilen, ohne Stetigkeitskorrektur. `p_fisher_l02_l04`: der p-Wert von Fishers exaktem Test.

In [ ]:
# TODO 8a
z_l02_l04 = None
p_z_l02_l04 = None

# TODO 8b
chi2_l02_l04 = None
p_fisher_l02_l04 = None

zeige(z_l02_l04, p_z_l02_l04, p_fisher_l02_l04)

In [ ]:
check("TODO 8a z-Test", lambda: math.isclose(z_l02_l04, 1.76877, abs_tol=1e-5) and math.isclose(p_z_l02_l04, 0.0769323, abs_tol=1e-6))
check("TODO 8b Chi-Quadrat gleich z hoch 2", lambda: math.isclose(float(chi2_l02_l04.statistic), z_l02_l04 ** 2, rel_tol=1e-9)
      and math.isclose(float(chi2_l02_l04.pvalue), p_z_l02_l04, rel_tol=1e-7))
check("TODO 8b Fisher", lambda: math.isclose(p_fisher_l02_l04, 0.0816041, abs_tol=1e-6))

L-02 liegt mit 0,76 % etwas über L-04 mit 0,68 %, aber $p = 0{,}077$: kein Nachweis bei $\alpha = 5\,\%$. Fisher kommt mit 0,082 auf fast dasselbe, bei so vielen Teilen trägt die Näherung. Die Annahme unabhängiger Teile bleibt: Mängel einer Lieferung können zusammenhängen.

**Chi-Quadrat-Unabhängigkeitstest.** Erwartete Häufigkeiten $E_{ij} = \text{Zeilensumme}_i \cdot \text{Spaltensumme}_j / n$, Teststatistik $\chi^2 = \sum (B - E)^2/E$ mit $(r - 1)(c - 1)$ Freiheitsgraden, standardisierte Residuen $(B - E)/\sqrt E$. Kategorie gegen Schicht wie im Skript:

In [ ]:
kat_schicht = pd.crosstab(stoerungen["kategorie"], stoerungen["schicht"])
chi2_ks = stats.chi2_contingency(kat_schicht)
erwartet_ks = pd.DataFrame(chi2_ks.expected_freq, index=kat_schicht.index, columns=kat_schicht.columns)
residuen_ks = (kat_schicht - erwartet_ks) / np.sqrt(erwartet_ks)
print(kat_schicht)
print(round(float(chi2_ks.statistic), 1), chi2_ks.dof, f"{float(chi2_ks.pvalue):.1e}", round(float(erwartet_ks.min().min()), 1))
print(residuen_ks.round(2))

### TODO 9 · Kategorie gegen Linie
`st_linie` verbindet jede Störung mit der Linie ihrer Maschine.
1. `kat_linie`: die Kreuztabelle von Kategorie (Zeilen) und Linie (Spalten). `chi2_kl`: das Ergebnis von `stats.chi2_contingency`. `erwartet_kl` und `residuen_kl`: die erwarteten Häufigkeiten und die standardisierten Residuen als DataFrames mit denselben Zeilen und Spalten wie `kat_linie`.
2. Zeichne in `fig_hm, ax_hm` die Residuen als Heatmap mit `ax_hm.imshow(...)` und einer divergierenden Farbskala wie `cmap="RdBu_r"`, symmetrisch um 0 (`vmin=-10, vmax=10`), mit beschrifteten Achsen und einer Farblegende (`fig_hm.colorbar`).

In [ ]:
st_linie = stoerungen.merge(maschinen[["maschine", "linie"]], on="maschine")

# TODO 9a
kat_linie = None
chi2_kl = None
erwartet_kl = None
residuen_kl = None

# TODO 9b
fig_hm, ax_hm = plt.subplots(figsize=(7, 4.5))

plt.show()
print(None if residuen_kl is None else residuen_kl.round(2))

In [ ]:
check("TODO 9a Tafel und Test", lambda: kat_linie.shape == (5, 4) and int(kat_linie.to_numpy().sum()) == 14263
      and math.isclose(float(chi2_kl.statistic), 386.067, abs_tol=1e-3) and int(chi2_kl.dof) == 12)
check("TODO 9a Residuen", lambda: residuen_kl.shape == (5, 4) and math.isclose(residuen_kl.loc["mechanisch", 1], 9.4476, abs_tol=1e-4)
      and math.isclose(residuen_kl.loc["Material", 4], 8.2313, abs_tol=1e-4) and math.isclose(erwartet_kl.loc["Bedienung", 1], 669.865, abs_tol=1e-3))
check("TODO 9b Heatmap", lambda: (len(ax_hm.images) >= 1 or any(type(c_).__name__ == "QuadMesh" for c_ in ax_hm.collections))
      and ax_hm.get_xlabel() != "" and ax_hm.get_ylabel() != "" and len(fig_hm.axes) >= 2)

Kategorie und Linie hängen stark zusammen, $\chi^2 = 386$ mit 12 Freiheitsgraden. Linie 1 mit den Drehmaschinen hat viel mehr mechanische Störungen als bei Unabhängigkeit erwartet (Residuum 9,4), Linie 4, die Endmontage, mehr Material- und Bedienungsfehler. Das ist wenig überraschend, denn die Linien arbeiten mit verschiedenen Verfahren. Die Residuen zeigen, wo die Abweichung sitzt.

## 5 · Teststärke und Stichprobengröße
*Lerneinheit 5 · Skript Abschnitt 5*

Die Teststärke $1 - \beta$ wächst mit dem Effekt $\delta$, mit $n$ und mit $\alpha$ und sinkt mit der Streuung $\sigma$. Für zwei gleich große Gruppen ist in der Normalnäherung

$$1 - \beta \approx \Phi\!\left(\frac{\delta}{\sigma\sqrt{2/n}} - z_{1-\alpha/2}\right), \qquad n \approx \frac{2\,(z_{1-\alpha/2} + z_{1-\beta})^2\,\sigma^2}{\delta^2}.$$

$\Phi$ ist `stats.norm.cdf`. Das Beispiel von Hand und die Lieferungen von L-05 aus dem Skript:

In [ ]:
z_a, z_b = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
print(round(z_a, 3), round(z_b, 3), round(2 * (z_a + z_b) ** 2 * 10 ** 2 / 5 ** 2, 1))

s_l01, s_l05, n_l01_lief = verzug_l01.std(), verzug_l05.std(), len(verzug_l01)
n_noetig = s_l05 ** 2 / ((3 / (z_a + z_b)) ** 2 - s_l01 ** 2 / n_l01_lief)
se_30 = math.sqrt(s_l05 ** 2 / 30 + s_l01 ** 2 / n_l01_lief)
staerke_30 = stats.norm.sf(z_a - 3 / se_30) + stats.norm.cdf(-z_a - 3 / se_30)
print(round(n_noetig, 1), round(staerke_30, 3), round((z_a + z_b) * se_30, 2))

### TODO 10 · Teststärke per Simulation
Die Instandhaltung will prüfen, ob eine neue Werkzeugeinstellung den Durchmesser um $\delta = 3$ µm verschiebt. Geplant sind zwei gleich große Gruppen, die Wellen streuen wie bei M-09, ausgewertet wird mit dem Welch-Test bei $\alpha = 5\,\%$.
1. `sigma_um`: die Standardabweichung von M-09 in µm. `n_formel`: die nötige Zahl je Gruppe für 80 % Teststärke nach der Formel.
2. `staerke_formel`: eine Series mit dem Index 25, 50, 100, 150 und der Teststärke nach der Normalnäherung oben, beide Ränder gezählt wie im Skript.
3. `staerke_sim`: dieselbe Series per Simulation. Ziehe für jedes $n$ 2.000 Wiederholungen beider Gruppen mit `rng.normal` als Matrizen der Form (2000, n), teste mit `stats.ttest_ind(..., axis=1, equal_var=False)` und zähle den Anteil mit $p < 0{,}05$.

In [ ]:
# TODO 10a
sigma_um = None
n_formel = None

# TODO 10b
staerke_formel = None

# TODO 10c
rng = np.random.default_rng(10)
staerke_sim = None

zeige(n_formel)
print(None if staerke_formel is None or staerke_sim is None else pd.DataFrame({"Formel": staerke_formel, "Simulation": staerke_sim}).round(4))

In [ ]:
STAERKE_REF = {25: 0.2625, 50: 0.4721, 100: 0.7661, 150: 0.9091}
check("TODO 10a Formel für n", lambda: math.isclose(sigma_um, 7.85889, abs_tol=1e-4) and math.isclose(n_formel, 107.725, abs_tol=0.01))
check("TODO 10b Teststärke nach Formel", lambda: list(staerke_formel.index) == [25, 50, 100, 150]
      and np.allclose(staerke_formel.to_numpy(), [0.27129, 0.47960, 0.77014, 0.91084], atol=1e-4))
check("TODO 10c Teststärke per Simulation", lambda: list(staerke_sim.index) == [25, 50, 100, 150]
      and all(abs(staerke_sim[n_] - c_) <= 4 * math.sqrt(c_ * (1 - c_) / 2000) + 0.005 for n_, c_ in STAERKE_REF.items()))

Für 80 % Teststärke braucht die Instandhaltung etwa 108 Wellen je Gruppe. Simulation und Formel liegen nah beieinander, die Formel ist bei kleinem $n$ etwas zu optimistisch, weil sie $\sigma$ als bekannt behandelt. Mit 25 Wellen je Gruppe würde ein echter Effekt von 3 µm in fast drei von vier Fällen übersehen.

## 6 · Viele Tests
*Lerneinheit 6 · Skript Abschnitt 6*

Bei $m$ Tests zum Niveau $\alpha$ steigt die Chance auf mindestens einen falschen Treffer auf bis zu $1 - (1 - \alpha)^m$.
- **Bonferroni:** jeden Test zum Niveau $\alpha/m$.
- **Holm:** p-Werte aufsteigend sortieren, $p_{(i)}$ mit $\alpha/(m - i + 1)$ vergleichen, beim ersten Misserfolg aufhören. Beide begrenzen die Familienfehlerrate (FWER).
- **Benjamini-Hochberg:** das größte $i$ mit $p_{(i)} \le i\,\alpha/m$ suchen und die Tests 1 bis $i$ verwerfen; begrenzt die Falscherkennungsrate (FDR). `stats.false_discovery_control(p)` gibt angepasste p-Werte zurück, die du mit $\alpha$ vergleichst.

Die 480 Maschinenmonate und die vier Linien aus dem Skript:

In [ ]:
schichten = produktion[["datum", "schicht"]].drop_duplicates()
schichten = schichten[pd.to_datetime(schichten["datum"]).dt.dayofweek < 5]
schichten_monat = schichten.groupby(schichten["datum"].str[:7]).size()
st_werktag = stoerungen[pd.to_datetime(stoerungen["datum"]).dt.dayofweek < 5]
zahl_mm = (st_werktag.groupby(["maschine", st_werktag["datum"].str[:7]]).size()
           .unstack(fill_value=0).reindex(columns=schichten_monat.index, fill_value=0))
rate_mm = zahl_mm.sum(axis=1) / schichten_monat.sum()
erwartet_mm = np.outer(rate_mm, schichten_monat)
k_mm = zahl_mm.to_numpy()
p_mm = np.minimum(1, 2 * np.minimum(stats.poisson.cdf(k_mm, erwartet_mm), stats.poisson.sf(k_mm - 1, erwartet_mm))).ravel()
print(zahl_mm.shape, int(schichten_monat.sum()), len(st_werktag), int((p_mm < 0.05).sum()), round(float(p_mm.min()), 4))
print(int((p_mm < 0.05 / len(p_mm)).sum()), int((stats.false_discovery_control(p_mm) < 0.05).sum()))

st_linie = stoerungen.merge(maschinen[["maschine", "linie"]], on="maschine")
p_linie = {}
for linie, gruppe in st_linie.groupby("linie"):
    tafel_linie = pd.crosstab(gruppe["schicht"] == "Nacht", gruppe["kategorie"] == "Bedienung")
    p_linie[linie] = float(stats.chi2_contingency(tafel_linie, correction=False).pvalue)
p_linie = pd.Series(p_linie)
print(p_linie.round(4).to_dict())
print(stats.false_discovery_control(p_linie).round(4))

### TODO 11 · Bedienungsfehler nachts, je Maschine
1. `p_maschine`: eine Series mit den 20 Maschinen als Index und dem p-Wert des Chi-Quadrat-Tests (ohne Stetigkeitskorrektur) für die 2×2-Tafel aus Nacht oder Tag und Bedienungsfehler oder nicht, wie oben je Linie.
2. `bonferroni(p, alpha=0.05)` und `holm(p, alpha=0.05)`: Funktionen, die zu einer Liste von p-Werten ein boolesches Array zurückgeben, `True` heißt verworfen. Die Reihenfolge entspricht der Eingabe. Rechne beide von Hand, ohne fertige Funktion.
3. `bh_maschine`: ein boolesches Array oder eine Series, `True` für die Maschinen, die nach Benjamini-Hochberg mit `stats.false_discovery_control` bei $\alpha = 0{,}05$ verworfen werden.

In [ ]:
# TODO 11a
p_maschine = None

# TODO 11b
def bonferroni(p, alpha=0.05):
    return None            # ersetze diese Zeile

def holm(p, alpha=0.05):
    return None            # ersetze diese Zeile

# TODO 11c
bh_maschine = None

print(None if p_maschine is None else p_maschine.round(4).to_dict())

In [ ]:
check("TODO 11a p-Werte je Maschine", lambda: len(p_maschine) == 20 and int((p_maschine < 0.05).sum()) == 6
      and math.isclose(float(p_maschine["M-04"]), 0.0038514, abs_tol=1e-6))
check("TODO 11b Bonferroni", lambda: list(bonferroni([0.004, 0.0152, 0.2131, 0.0002])) == [True, False, False, True]
      and not any(bonferroni(p_maschine)))
check("TODO 11b Holm", lambda: list(holm([0.004, 0.0152, 0.2131, 0.0002])) == [True, True, False, True]
      and list(holm([0.03, 0.2, 0.4])) == [False, False, False] and list(holm([0.001, 0.03, 0.04, 0.045])) == [True, False, False, False]
      and list(holm([0.02, 0.03, 0.04, 0.045])) == [False] * 4 and not any(holm(p_maschine)))
check("TODO 11c Benjamini-Hochberg", lambda: len(bh_maschine) == 20 and not any(np.asarray(bh_maschine)))

In [ ]:
if p_maschine is None:
    print("⏳ Erst TODO 11 lösen.")
else:
    print("p unter 0,05:", ", ".join(f"{m_} ({p_:.4f})" for m_, p_ in p_maschine[p_maschine < 0.05].sort_values().items()))
    print(f"kleinster p-Wert mal 20: {min(1.0, 20 * p_maschine.min()):.3f}")

Sechs Maschinen haben $p < 0{,}05$, nach keiner Korrektur bleibt eine übrig: Der kleinste p-Wert, 0,0039 bei M-04, wird mit 20 malgenommen 0,077. Je Linie überstanden drei von vier Tests die Korrektur. Wer die Daten auf 20 Maschinen verteilt, hat in jedem Test weniger Störungen und damit weniger Teststärke, und die Familie wird größer. Die Holm-Tests mit den kleinen Listen prüfen die typischen Fehler: falsch herum sortiert, nicht aufgehört, von oben nach unten gesucht.

## 7 · Signifikant ist nicht relevant
*Lerneinheit 7 · Skript Abschnitt 7*

Mit vielen Daten wird jede Abweichung signifikant. Ob sie zählt, zeigen der Schätzwert mit Intervall und eine Effektstärke wie **Cohen d** $= (\bar x - \mu_0)/s$, geteilt durch die Streuung der Einzelwerte. Den Nachweis, dass ein Mittelwert innerhalb von Grenzen $\mu_0 \pm \Delta$ liegt, führt der **Äquivalenztest** TOST: zwei einseitige t-Tests gegen $\mu_0 - \Delta$ (`alternative="greater"`) und gegen $\mu_0 + \Delta$ (`alternative="less"`), der größere p-Wert entscheidet. M-03 mit allen Messungen und der TOST für M-09 wie im Skript:

In [ ]:
mu_m03, sigma_m03 = m03.mean(), m03.std()
test_m03 = stats.ttest_1samp(m03, 25.0)
ki_m03 = test_m03.confidence_interval()
d_m03 = (m03.mean() - 25.0) / m03.std()
print(f"{float(test_m03.pvalue):.1e}", round(float(test_m03.statistic), 2), round(1000 * (float(ki_m03.low) - 25), 2), round(1000 * (float(ki_m03.high) - 25), 2), round(d_m03, 2))
aussen_ist = stats.norm.cdf(24.975, mu_m03, sigma_m03) + stats.norm.sf(25.025, mu_m03, sigma_m03)
aussen_mitte = 2 * stats.norm.cdf(24.975, 25.0, sigma_m03)
print(round(float(aussen_ist), 4), round(float(aussen_mitte), 4))

for name, messung in [("Woche", woche.loc[woche["maschine"] == "M-09", "durchmesser_mm"]), ("alle", m09)]:
    p_unten = stats.ttest_1samp(messung, 25.0 - 0.002, alternative="greater").pvalue
    p_oben = stats.ttest_1samp(messung, 25.0 + 0.002, alternative="less").pvalue
    ki90 = stats.ttest_1samp(messung, 25.0).confidence_interval(confidence_level=0.90)
    print(name, len(messung), f"{float(p_unten):.2g}", f"{float(p_oben):.2g}", round(1000 * (float(ki90.low) - 25), 2), round(1000 * (float(ki90.high) - 25), 2))

### TODO 12 · Relevanz bei M-03 und M-09
Du brauchst `kw3` aus TODO 3.
1. `d_m09`: Cohen d für M-09 mit allen Messungen gegen 25,000 mm. `d_kw3`: Cohen d für M-03 in der Januarwoche (`kw3`).
2. `aussen_sigma09`: der Anteil der Wellen außerhalb der Toleranz 24,975 bis 25,025 mm nach der Normalverteilung, wenn M-03 ihren Mittelwert `mu_m03` behält, aber so wenig streut wie M-09.
3. `tost_m03` und `tost_kw3`: der p-Wert des TOST für die Grenzen ±5 µm um 25,000 mm, für M-03 mit allen Messungen und für die Januarwoche.

In [ ]:
# TODO 12a
d_m09 = None
d_kw3 = None

# TODO 12b
aussen_sigma09 = None

# TODO 12c
tost_m03 = None
tost_kw3 = None

zeige(d_m09, d_kw3, aussen_sigma09, tost_m03, tost_kw3)

In [ ]:
check("TODO 12a Cohen d", lambda: math.isclose(d_m09, -0.0031518, abs_tol=1e-6) and math.isclose(d_kw3, 0.367206, abs_tol=1e-5))
check("TODO 12b Toleranzanteil", lambda: math.isclose(aussen_sigma09, 0.0036456, abs_tol=1e-6))
check("TODO 12c TOST", lambda: math.isclose(tost_m03, 1.54295e-07, rel_tol=1e-4) and math.isclose(tost_kw3, 0.242708, abs_tol=1e-5))

Mit $d \approx 0$ liegt M-09 praktisch genau auf Sollmaß. Die Januarwoche von M-03 hat $d = 0{,}37$, ähnlich wie alle Messungen mit 0,31: $d$ hängt kaum von $n$ ab. Würde M-03 so wenig streuen wie M-09, lägen trotz der Abweichung von 3,8 µm nur 0,36 % der Wellen außerhalb der Toleranz statt 5,0 %, zentriert wären es 4,0 %. Die Streuung ist also das größere Problem. Mit allen Messungen ist M-03 zugleich nachweislich vom Sollmaß verschieden und nachweislich innerhalb von ±5 µm. Mit einer Woche ist die Äquivalenz nicht gezeigt, $p = 0{,}24$.

### TODO 13 · Ein vollständiger Ergebnissatz
`bericht(a, b, name_a, name_b)` vergleicht die Verzüge zweier Lieferanten mit dem Welch-Test und gibt ein Dictionary mit den Schlüsseln `differenz` ($\bar x_a - \bar x_b$), `unten` und `oben` (95-%-Intervall der Differenz aus `confidence_interval()`), `t`, `df`, `p`, `d` (Cohen d mit der gemeinsamen Standardabweichung $s_p$ aus dem Skript), `n_a`, `n_b` und `satz` zurück. `satz` ist ein deutscher Satz mit allen Angaben, Zahlen mit Dezimalkomma, etwa „L-02 kam im Mittel 0,59 h später als L-01 (95-%-Intervall …)“.

In [ ]:
# TODO 13
def bericht(a, b, name_a, name_b):
    return None            # ersetze diese Zeile

verzug_l02 = lieferungen.loc[lieferungen["lieferant"] == "L-02", "verzug_h"]
verzug_l04 = lieferungen.loc[lieferungen["lieferant"] == "L-04", "verzug_h"]
bericht_l02 = bericht(verzug_l02, verzug_l01, "L-02", "L-01")
bericht_l04 = bericht(verzug_l04, verzug_l01, "L-04", "L-01")
for b_ in [bericht_l02, bericht_l04]:
    print(None if b_ is None else b_["satz"])

In [ ]:
import re

SCHLUESSEL = {"differenz", "unten", "oben", "t", "df", "p", "d", "n_a", "n_b", "satz"}
check("TODO 13 Schlüssel", lambda: SCHLUESSEL <= set(bericht_l02) and SCHLUESSEL <= set(bericht_l04))
check("TODO 13 L-02 gegen L-01", lambda: math.isclose(bericht_l02["differenz"], 0.588431, abs_tol=1e-5)
      and math.isclose(bericht_l02["unten"], -0.560142, abs_tol=1e-5) and math.isclose(bericht_l02["oben"], 1.737004, abs_tol=1e-5)
      and math.isclose(bericht_l02["p"], 0.315147, abs_tol=1e-5) and math.isclose(bericht_l02["d"], 0.0440269, abs_tol=1e-5)
      and (bericht_l02["n_a"], bericht_l02["n_b"]) == (967, 1179))
check("TODO 13 L-04 gegen L-01", lambda: math.isclose(bericht_l04["t"], -10.8424, abs_tol=1e-4) and math.isclose(bericht_l04["df"], 1565.92, abs_tol=0.01)
      and math.isclose(bericht_l04["d"], -0.438999, abs_tol=1e-5))
check("TODO 13 Satz", lambda: all(isinstance(b_["satz"], str) and "L-01" in b_["satz"] and "Welch" in b_["satz"]
                                 and re.search(r"\d,\d", b_["satz"]) and not re.search(r"\d\.\d{1,2}(?!\d)", b_["satz"]) for b_ in [bericht_l02, bericht_l04]))

L-02 kam im Mittel 0,59 Stunden später als L-01, das Intervall reicht von −0,56 bis 1,74 Stunden, $p = 0{,}32$ und $d = 0{,}04$: kein Nachweis, und auch der größte plausible Unterschied ist klein. L-04 kam 4,36 Stunden früher, $p < 0{,}001$, $d = −0{,}44$. Ein solcher Satz enthält alles, was der Einkauf braucht, und lässt sich nachrechnen.

## Bonus

### Bonus B1 · Immer wieder hineinschauen
Zwei Gruppen werden aus denselben Verzügen von L-01 gezogen, es gibt also keinen echten Unterschied. Wer nach jeweils 50 weiteren Lieferungen je Gruppe testet und beim ersten $p < 0{,}05$ aufhört, meldet viel öfter einen Unterschied als 5 %.
1. Ziehe mit `rng` 2.000 Wiederholungen von je 500 Lieferungen je Gruppe mit Zurücklegen aus `verzug_l01` (zwei Matrizen der Form (2000, 500)).
2. `anteil_stopp`: der Anteil der Wiederholungen, in denen der Welch-Test nach 50, 100, …, 500 Lieferungen je Gruppe mindestens einmal $p < 0{,}05$ liefert. `anteil_fest`: der Anteil mit $p < 0{,}05$, wenn nur einmal nach 500 Lieferungen getestet wird.

In [ ]:
# Bonus B1
rng = np.random.default_rng(101)
anteil_stopp = None
anteil_fest = None

zeige(anteil_stopp, anteil_fest)

In [ ]:
check("B1 Hineinschauen", lambda: 0.15 <= anteil_stopp <= 0.235 and 0.025 <= anteil_fest <= 0.075)

Mit zehnmal Hineinschauen meldet etwa jede fünfte Wiederholung einen Unterschied, der nicht existiert. Mit einem einzigen Test am Ende bleibt es bei etwa 5 %. Die Zahl der Lieferungen und der Zeitpunkt des Tests gehören deshalb vorher festgelegt.

### Bonus B2 · Der Fluch des Gewinners
Eine neue Einstellung verschiebt den Durchmesser um 2 µm, getestet wird mit nur 20 Wellen je Gruppe, die Wellen streuen wie bei M-09 mit `1000 * m09.std()` µm.
1. Simuliere 10.000 Studien mit `rng.normal` und werte jede mit dem Welch-Test aus.
2. `staerke_b2`: der Anteil signifikanter Studien. `mittel_signif_b2`: der mittlere geschätzte Unterschied unter den signifikanten Studien in µm. `uebertreibung_b2`: `mittel_signif_b2 / 2`.

In [ ]:
# Bonus B2
rng = np.random.default_rng(102)
staerke_b2 = None
mittel_signif_b2 = None
uebertreibung_b2 = None

zeige(staerke_b2, mittel_signif_b2, uebertreibung_b2)

In [ ]:
check("B2 Fluch des Gewinners", lambda: 0.10 <= staerke_b2 <= 0.145 and 2.3 <= uebertreibung_b2 <= 3.4)

Nur etwa jede achte Studie ist signifikant, und diese zeigen im Mittel fast das Dreifache des wahren Effekts. Ein signifikantes Ergebnis aus einer kleinen Studie ist deshalb ein Anlass, genauer nachzumessen, keine verlässliche Schätzung der Größe.

## Zum Nachdenken

1. In TODO 1 hatte der verzerrte Schätzer den kleineren mittleren quadratischen Fehler. Warum nimmt man für Intervalle trotzdem $s^2$ mit $n - 1$?
2. In TODO 3 entscheidet die Wahl zwischen $z$ und $t$, ob das Sollmaß im Intervall liegt. Wie würdest du das Ergebnis dieser Woche der Instandhaltung berichten?
3. In TODO 5 weicht M-03 nur in 33 von 101 Wochen nachweisbar ab. Was würdest du der Instandhaltung für ihre wöchentliche Prüfung raten?
4. Die Teile einer Lieferung sind nicht unabhängig, wenn eine ganze Charge schlecht ist. Wie würde sich das auf den p-Wert in TODO 8 auswirken, und wie könntest du es berücksichtigen?
5. Der Einkauf möchte L-05 auslisten, wenn es „signifikant später“ kommt. Was schlägst du vor, bevor überhaupt getestet wird?

<details><summary>Hinweise zu den Antworten</summary>

1. Die t-Verteilung und damit die Intervalle und Tests sind für $s^2$ mit $n - 1$ hergeleitet. Mit dem verzerrten Schätzer würden die Intervalle zu schmal und träfen seltener als versprochen. Der kleinere MSE hilft, wenn eine einzelne Schätzung möglichst nah liegen soll, nicht beim Bauen von Intervallen.
2. Mit Intervall und exaktem p-Wert: „Mittelwert 25,0037 mm, 95-%-Intervall 24,99994 bis 25,00746 mm, $p = 0{,}054$“. Wichtiger als die Schwelle ist, dass die Abweichung von 3,7 µm zu dem passt, was alle Messungen von M-03 zeigen.
3. Eine Woche mit 30 Messungen hat für eine Abweichung von 3,8 µm nur etwa 40 % Teststärke. Besser sind Daten über mehrere Wochen, eine Regelkarte oder ein vorher geplanter Stichprobenumfang. Und vorher klären, welche Abweichung überhaupt ein Eingreifen rechtfertigt.
4. Die Mängel streuen dann stärker, als die Binomialverteilung annimmt, der Standardfehler ist zu klein und der p-Wert zu klein. Abhilfe: ganze Lieferungen als Einheit nehmen, etwa mit einem Permutationstest, der Lieferungen mischt, oder mit den Quoten je Lieferung.
5. Festlegen, welcher Unterschied für den Einkauf zählt, welches $\alpha$ und welche Teststärke gelten sollen, und daraus die nötige Zahl der Lieferungen ausrechnen. Mit 30 Lieferungen ist ein Unterschied von 3 Stunden kaum nachweisbar, ein Nein des Tests wäre also kein Freispruch.
</details>

## Wie es weitergeht

Du hast Schätzer beurteilt, Intervalle und Tests mit Formeln gerechnet, die Teststärke geplant und viele Tests korrigiert. Bisher ging es um eine Größe gegen einen Sollwert oder um zwei Gruppen. **Modul 7 „Lineare Regression“** fragt, wie eine Zielgröße von mehreren Einflüssen gleichzeitig abhängt, etwa der Verzug vom Lieferanten, der Menge und dem Wochentag, und nutzt dabei das Prinzip der Maximum Likelihood aus Abschnitt 1.